# Adaptive Subject-Level Differential Privacy — Colab Runner

**Before running:** Go to `Runtime → Change runtime type → T4 GPU`

This notebook clones the repo, installs dependencies, and runs the full experiment.

## 1. Clone the repo

In [5]:
!git clone https://github.com/shaktsin/ml-research.git
%cd ml-research/adaptive-diff-privacy

Cloning into 'ml-research'...
remote: Enumerating objects: 21, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 21 (delta 4), reused 18 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (21/21), 15.50 KiB | 15.50 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/ml-research/adaptive-diff-privacy/ml-research/adaptive-diff-privacy


## 2. Install dependencies

In [6]:
!pip install -q -r requirements.txt

## 3. Verify GPU

In [7]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: Tesla T4


## 4. Run the experiment

Runs all 3 configs: Baseline, Subject-DP (uniform), Adaptive Subject-DP.

In [8]:
!python run_experiment.py

Device: cuda
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 222kB/s]
vocab.txt: 232kB [00:00, 3.16MB/s]
tokenizer.json: 466kB [00:00, 3.56MB/s]
Loading AG News dataset...
README.md: 8.07kB [00:00, 26.0MB/s]
data/train-00000-of-00001.parquet: 100% 18.6M/18.6M [00:01<00:00, 15.0MB/s]
data/test-00000-of-00001.parquet: 100% 1.23M/1.23M [00:00<00:00, 2.02MB/s]
Generating train split: 100% 120000/120000 [00:00<00:00, 323740.73 examples/s]
Generating test split: 100% 7600/7600 [00:00<00:00, 688794.28 examples/s]

  Config: Baseline (no DP)
config.json: 100% 483/483 [00:00<00:00, 2.68MB/s]
model.safetensors: 100% 268M/268M [00:01<00:00, 190MB/s]
Loading weights: 100% 100/100 [00:00<00:00, 1199.56it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weig

## 5. (Optional) Tweak parameters and re-run

Edit the knobs below and re-run a single config without re-running everything.

In [ ]:
import sys
sys.path.insert(0, '.')

from data import get_dataloaders
from model import get_model
from trainer import train_baseline, train_subject_dp, evaluate
from mia import run_mia

# --- Knobs ---
EPOCHS     = 3
CLIP_NORM  = 1.0
BASE_SIGMA = 1.0
N_SUBJECTS = 500
MAX_TRAIN  = 2000   # set None for full 120k dataset (much slower)
BATCH_SIZE = 16

train_loader, test_loader = get_dataloaders(
    max_train=MAX_TRAIN, batch_size=BATCH_SIZE, n_subjects=N_SUBJECTS
)

print('Data loaded.')
print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

In [ ]:
# Run only Adaptive Subject-DP
model = get_model(device=device)
train_subject_dp(model, train_loader, epochs=EPOCHS,
                 clip_norm=CLIP_NORM, base_sigma=BASE_SIGMA, adaptive=True)
acc = evaluate(model, test_loader)
auc = run_mia(model, train_loader, test_loader)
print(f'Accuracy: {acc:.4f} | MIA AUC: {auc:.4f}')